# New Heuristic Test: Load Before, Offset Matrix, Result

This notebook tests the updated lower heuristic. For each fixed global bias matrix, it shows:

1. load matrix before control
2. applied A3 offset matrices produced by the heuristic
3. load matrix after control

It also shows UE-count changes and handovers so we can separate real mobility effects from radio/queue load drift.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "global_ppo_3gnb_env.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from global_ppo_3gnb_env import GlobalPPO3GNBEnv, SLICE_TYPES

plt.rcParams.update({"figure.dpi": 125, "axes.grid": True, "grid.alpha": 0.22})
np.set_printoptions(precision=3, suppress=True)

GNB_IDS = [0, 1, 2]
print(f"Project root: {PROJECT_ROOT}")
print("Slices:", SLICE_TYPES)

## Scenario

Default example: gNB0 has eMBB load `0.80`; the other cells are light. UE counts are fixed to 5 per slice, 15 total.

In [ ]:
SCENARIO_NAME = "embb_g0_offload"
TARGET_LOAD_MATRIX = np.array([
    [0.80, 0.20, 0.20],
    [0.20, 0.20, 0.20],
    [0.20, 0.20, 0.20],
], dtype=float)

CUSTOM_UE_COUNTS = pd.DataFrame(
    [[3, 1, 1], [1, 2, 2], [1, 2, 2]],
    index=[f"gNB {g}" for g in GNB_IDS],
    columns=SLICE_TYPES,
)

N_WINDOWS = 3
LOCAL_STEPS_PER_GLOBAL = 10
MAX_HANDOVERS_PER_LOCAL_STEP = 1
SEED = 7

display(pd.DataFrame(TARGET_LOAD_MATRIX, index=CUSTOM_UE_COUNTS.index, columns=SLICE_TYPES).style.set_caption("Target load matrix"))
display(CUSTOM_UE_COUNTS.style.set_caption(f"Custom UE counts, total={int(CUSTOM_UE_COUNTS.to_numpy().sum())}"))

## Fixed Bias Tests

Small values below the heuristic deadband should behave neutral. Clear negative eMBB bias should produce negative A3 offsets for gNB0/eMBB.

In [ ]:
def g0_embb_bias(strength):
    strength = float(strength)
    return np.array([
        [-strength, 0.0, 0.0],
        [ strength, 0.0, 0.0],
        [ strength, 0.0, 0.0],
    ], dtype=float)

BIAS_TESTS = {
    "neutral_zero": np.zeros((3, 3), dtype=float),
    "tiny_0.20_deadband": g0_embb_bias(0.20),
    "soft_0.30_offload": g0_embb_bias(0.30),
    "medium_0.50_offload": g0_embb_bias(0.50),
    "strong_1.00_offload": g0_embb_bias(1.00),
}

for name, matrix in BIAS_TESTS.items():
    print(name)
    display(pd.DataFrame(matrix, index=[f"gNB {g}" for g in GNB_IDS], columns=SLICE_TYPES))

## Helpers

In [ ]:
def matrix_df(matrix):
    return pd.DataFrame(np.asarray(matrix, dtype=float), index=[f"gNB {g}" for g in GNB_IDS], columns=SLICE_TYPES)

def heatmap(ax, matrix, title, cmap="viridis", vmin=0.0, vmax=1.0, fmt=".2f"):
    matrix = np.asarray(matrix, dtype=float)
    ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(SLICE_TYPES)), SLICE_TYPES)
    ax.set_yticks(range(len(GNB_IDS)), [f"gNB {g}" for g in GNB_IDS])
    ax.set_title(title)
    for r in range(matrix.shape[0]):
        for c in range(matrix.shape[1]):
            ax.text(c, r, format(matrix[r, c], fmt), ha="center", va="center", color="white", fontsize=8, fontweight="bold")

def make_env(seed=SEED):
    return GlobalPPO3GNBEnv(
        seed=int(seed),
        n_gnbs=3,
        include_ue_counts=True,
        use_sumo_mobility=False,
        local_steps_per_global=LOCAL_STEPS_PER_GLOBAL,
        global_steps_per_episode=N_WINDOWS,
        scenario_mode="snapshot",
        snapshot_scenario=SCENARIO_NAME,
        terminal_reward_only=False,
        use_progress_reward=False,
        max_handovers_per_local_step=MAX_HANDOVERS_PER_LOCAL_STEP,
        action_direction_reward_weight=0.0,
        snapshot_block_episodes=10,
        light_load_ues=1,
        medium_load_ues=2,
        high_load_ues=3,
    )

def reset_custom_scenario(env):
    obs, info = env.reset()
    env.base_env.clear_ues(reset_ids=True)
    env._active_scenario = f"{SCENARIO_NAME}_custom"
    env._active_target_load_matrix = TARGET_LOAD_MATRIX.copy()
    counts = CUSTOM_UE_COUNTS.to_numpy(dtype=int)
    for gnb_id in GNB_IDS:
        for s_idx, slice_type in enumerate(SLICE_TYPES):
            target = float(TARGET_LOAD_MATRIX[gnb_id, s_idx])
            for ue_idx in range(int(counts[gnb_id, s_idx])):
                x, y = env._sample_ue_position(gnb_id, slice_type, target, ue_idx)
                ue_id = env.base_env.add_ue(x=x, y=y, vx=0.0, vy=0.0, slice_type=slice_type)
                env._force_attach(ue_id, gnb_id)
            env._set_slice_prb_load(gnb_id, slice_type, target)
    env.base_env._invalidate_metric_caches()
    obs = env._get_observation()
    err = env._target_load_error()
    info = env._build_info(reward=0.0, instant_rewards=[], handovers=0, start_imbalance=err, end_imbalance=err)
    return obs, info

def ue_count_matrix(env):
    return np.asarray([[env.base_env.get_slice_ue_count(g, s) for s in SLICE_TYPES] for g in GNB_IDS], dtype=float)

def prb_used_matrix(env):
    return np.asarray([[env.base_env.get_slice_used_prbs(g, s) for s in SLICE_TYPES] for g in GNB_IDS], dtype=float)

def prb_budget_matrix(env):
    return np.asarray([[env.base_env.get_slice_prb_budget(g, s) for s in SLICE_TYPES] for g in GNB_IDS], dtype=float)

## Offset Matrices

For each target neighbor, the matrix entry `(gNB, slice)` is the applied A3 offset from that serving gNB to the target neighbor. Rows where serving equals target are blank/NaN.

In [ ]:
def collect_offsets(env, bias_matrix):
    ue_counts = env._ue_count_dict()
    slice_loads = env._slice_load_dict()
    kmax = env._kmax_by_slice()
    rows = []
    for gnb_id, agent in env.lower_agents.items():
        bias_row = {s: float(bias_matrix[gnb_id, s_idx]) for s_idx, s in enumerate(SLICE_TYPES)}
        offsets = agent.compute_offsets(
            bias_row=bias_row,
            ue_counts=ue_counts,
            kmax=kmax,
            slice_loads=slice_loads,
            handover_failure_ratios={},
            ping_pong_ratios={},
        )
        for (src, dst, slice_type), value in offsets.items():
            rows.append({
                "from_gnb": src,
                "to_gnb": dst,
                "slice": slice_type,
                "bias": bias_row[slice_type],
                "proto_offset_db": float(value["proto_offset_db"]),
                "applied_offset_db": float(value["applied_offset_db"]),
                "bias_term_db": float(value.get("bias_term_db", np.nan)),
                "safety_term_db": float(value.get("safety_term_db", np.nan)),
                "neighbor_ue_fraction": float(value.get("neighbor_ue_fraction", np.nan)),
            })
    return pd.DataFrame(rows).sort_values(["from_gnb", "slice", "to_gnb"])

def offset_matrix_for_target(offsets_df, target_gnb, value_col="applied_offset_db"):
    matrix = np.full((len(GNB_IDS), len(SLICE_TYPES)), np.nan, dtype=float)
    for _, row in offsets_df[offsets_df["to_gnb"] == target_gnb].iterrows():
        src = int(row["from_gnb"])
        s_idx = list(SLICE_TYPES).index(row["slice"])
        matrix[src, s_idx] = float(row[value_col])
    return matrix

## Run One Test

In [ ]:
def run_test(test_name, bias_matrix):
    env = make_env()
    try:
        obs, info = reset_custom_scenario(env)
        load_before = env._load_matrix().copy()
        ue_before = ue_count_matrix(env)
        used_before = prb_used_matrix(env)
        budget = prb_budget_matrix(env)
        offsets_before = collect_offsets(env, bias_matrix)

        total_handovers = 0
        step_rows = []
        for window in range(N_WINDOWS):
            before = env._load_matrix().copy()
            obs, reward, terminated, truncated, info = env.step(bias_matrix.reshape(-1))
            after = np.asarray(info["load_matrix"], dtype=float)
            total_handovers += int(info["handover_count"])
            step_rows.append({
                "window": window,
                "reward": float(reward),
                "handover_count": int(info["handover_count"]),
                "target_load_error": float(info["target_load_error"]),
                "sla_count": float(info["sla_count"]),
                "mean_abs_load_change": float(np.mean(np.abs(after - before))),
            })
            if terminated or truncated:
                break

        load_after = env._load_matrix().copy()
        ue_after = ue_count_matrix(env)
        used_after = prb_used_matrix(env)
    finally:
        env.close()

    return {
        "test": test_name,
        "bias_matrix": np.asarray(bias_matrix, dtype=float),
        "load_before": load_before,
        "load_after": load_after,
        "load_delta": load_after - load_before,
        "ue_before": ue_before,
        "ue_after": ue_after,
        "ue_delta": ue_after - ue_before,
        "used_before": used_before,
        "used_after": used_after,
        "budget": budget,
        "offsets": offsets_before,
        "steps": pd.DataFrame(step_rows),
        "total_handovers": total_handovers,
    }

TEST_NAME = "soft_0.30_offload"
result = run_test(TEST_NAME, BIAS_TESTS[TEST_NAME])

print(f"test: {TEST_NAME}")
print(f"total handovers: {result['total_handovers']}")
display(result["steps"])

## Load Before → Offset Matrices → Result

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4), constrained_layout=True)
heatmap(axes[0], result["load_before"], "load before", cmap="viridis", vmin=0, vmax=1)
heatmap(axes[1], result["bias_matrix"], "fixed bias", cmap="coolwarm", vmin=-1, vmax=1)
heatmap(axes[2], result["load_after"], "load after", cmap="viridis", vmin=0, vmax=1)
heatmap(axes[3], result["load_delta"], "load delta", cmap="coolwarm", vmin=-1, vmax=1)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
for idx, target_gnb in enumerate(GNB_IDS):
    heatmap(
        axes[idx],
        offset_matrix_for_target(result["offsets"], target_gnb),
        f"applied offsets to gNB {target_gnb}",
        cmap="coolwarm",
        vmin=-6,
        vmax=6,
        fmt=".0f",
    )
plt.show()

display(result["offsets"])

fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
heatmap(axes[0], result["ue_before"], "UE count before", cmap="magma", vmin=0, vmax=max(1, result["ue_before"].max()))
heatmap(axes[1], result["ue_after"], "UE count after", cmap="magma", vmin=0, vmax=max(1, result["ue_after"].max()))
heatmap(axes[2], result["ue_delta"], "UE count delta", cmap="coolwarm", vmin=-3, vmax=3)
plt.show()

## PRB Diagnostic

Load is `used PRBs / PRB budget`, so this section explains why load can stay high even after UE count changes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
heatmap(axes[0], result["used_before"], "used PRBs before", cmap="viridis", vmin=0, vmax=max(1, result["used_before"].max(), result["used_after"].max()))
heatmap(axes[1], result["used_after"], "used PRBs after", cmap="viridis", vmin=0, vmax=max(1, result["used_before"].max(), result["used_after"].max()))
heatmap(axes[2], result["budget"], "PRB budget", cmap="viridis", vmin=0, vmax=max(1, result["budget"].max()))
plt.show()

print("load before = used before / budget, clipped to [0,1]")
display(matrix_df(result["load_before"]))
print("load after = used after / budget, clipped to [0,1]")
display(matrix_df(result["load_after"]))

## Balance Load Result

This summarizes whether the heuristic moved the final load closer to the balanced per-slice target. The balance target is the mean target load for each slice repeated across gNBs.

In [ ]:
def balance_target_matrix(target_matrix):
    per_slice_target = np.mean(np.asarray(target_matrix, dtype=float), axis=0, keepdims=True)
    return np.repeat(per_slice_target, len(GNB_IDS), axis=0)

def balance_metrics(load_matrix, balance_target):
    load_matrix = np.asarray(load_matrix, dtype=float)
    balance_target = np.asarray(balance_target, dtype=float)
    return {
        "target_load_error": float(np.mean((load_matrix - balance_target) ** 2)),
        "load_variance_sum": float(sum(np.var(load_matrix[:, s_idx]) for s_idx in range(len(SLICE_TYPES)))),
        "mean_abs_balance_error": float(np.mean(np.abs(load_matrix - balance_target))),
        "max_abs_balance_error": float(np.max(np.abs(load_matrix - balance_target))),
    }

balance_target = balance_target_matrix(TARGET_LOAD_MATRIX)
before_metrics = balance_metrics(result["load_before"], balance_target)
after_metrics = balance_metrics(result["load_after"], balance_target)

balance_summary = pd.DataFrame([
    {"stage": "before", **before_metrics},
    {"stage": "after", **after_metrics},
])
improvement = before_metrics["target_load_error"] - after_metrics["target_load_error"]

fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
heatmap(axes[0], balance_target, "balance target", cmap="viridis", vmin=0, vmax=1)
heatmap(axes[1], result["load_before"] - balance_target, "before - target", cmap="coolwarm", vmin=-1, vmax=1)
heatmap(axes[2], result["load_after"] - balance_target, "after - target", cmap="coolwarm", vmin=-1, vmax=1)
plt.show()

display(balance_summary)
print(f"target-load-error improvement: {improvement:.4f}")
print("balanced better after?", improvement > 0.0)

## Run All Bias Tests

In [ ]:
all_results = {name: run_test(name, matrix) for name, matrix in BIAS_TESTS.items()}
summary_rows = []
for name, res in all_results.items():
    summary_rows.append({
        "test": name,
        "total_handovers": res["total_handovers"],
        "g0_eMBB_load_before": float(res["load_before"][0, 0]),
        "g0_eMBB_load_after": float(res["load_after"][0, 0]),
        "g0_eMBB_ue_before": float(res["ue_before"][0, 0]),
        "g0_eMBB_ue_after": float(res["ue_after"][0, 0]),
        "mean_abs_load_delta": float(np.mean(np.abs(res["load_delta"]))),
        "final_target_error": float(res["steps"]["target_load_error"].iloc[-1]),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

## Search For Target eMBB Balance

This searches simple one-window and two-window bias schedules to get close to your desired eMBB load vector: `gNB0 ≈ 0.50`, `gNB1 ≈ 0.35`, `gNB2 ≈ 0.35`.

In [ ]:
DESIRED_EMBB_LOAD = np.array([0.50, 0.35, 0.35], dtype=float)

def run_schedule(schedule, seed=SEED):
    env = make_env(seed)
    try:
        obs, info = reset_custom_scenario(env)
        initial_load = env._load_matrix().copy()
        initial_ue = ue_count_matrix(env)
        total_handovers = 0
        for window, bias_matrix in enumerate(schedule):
            obs, reward, terminated, truncated, info = env.step(np.asarray(bias_matrix, dtype=float).reshape(-1))
            total_handovers += int(info["handover_count"])
            if terminated or truncated:
                break
        final_load = env._load_matrix().copy()
        final_ue = ue_count_matrix(env)
    finally:
        env.close()
    return initial_load, final_load, initial_ue, final_ue, total_handovers

def make_bias(g0_strength, g1_strength=0.0, g2_strength=0.0):
    return np.array([
        [-float(g0_strength), 0.0, 0.0],
        [ float(g1_strength), 0.0, 0.0],
        [ float(g2_strength), 0.0, 0.0],
    ], dtype=float)

candidate_schedules = []
for s0 in [0.0, 0.20, 0.26, 0.30, 0.35, 0.40, 0.50, 0.60]:
    for s1 in [0.0, 0.20, 0.30, 0.50, 1.00]:
        for s2 in [0.0, 0.20, 0.30, 0.50, 1.00]:
            B = make_bias(s0, s1, s2)
            candidate_schedules.append((f"one_win_g0{s0:.2f}_g1{s1:.2f}_g2{s2:.2f}", [B]))
            candidate_schedules.append((f"two_win_g0{s0:.2f}_g1{s1:.2f}_g2{s2:.2f}", [B, B]))
            candidate_schedules.append((f"pulse_then_stop_g0{s0:.2f}_g1{s1:.2f}_g2{s2:.2f}", [B, np.zeros((3, 3))]))

search_rows = []
for name, schedule in candidate_schedules:
    initial_load, final_load, initial_ue, final_ue, total_handovers = run_schedule(schedule)
    embb_final = final_load[:, 0]
    score = float(np.mean((embb_final - DESIRED_EMBB_LOAD) ** 2))
    search_rows.append({
        "schedule": name,
        "score": score,
        "handovers": int(total_handovers),
        "g0_eMBB_load": float(embb_final[0]),
        "g1_eMBB_load": float(embb_final[1]),
        "g2_eMBB_load": float(embb_final[2]),
        "g0_eMBB_ue": float(final_ue[0, 0]),
        "g1_eMBB_ue": float(final_ue[1, 0]),
        "g2_eMBB_ue": float(final_ue[2, 0]),
    })

search_df = pd.DataFrame(search_rows).sort_values(["score", "handovers"]).reset_index(drop=True)
display(search_df.head(20))

best_schedule_name = search_df.iloc[0]["schedule"]
best_schedule = dict(candidate_schedules)[best_schedule_name]
best_initial_load, best_final_load, best_initial_ue, best_final_ue, best_handovers = run_schedule(best_schedule)

print("best schedule:", best_schedule_name)
print("desired eMBB:", DESIRED_EMBB_LOAD.tolist())
print("final eMBB:", np.round(best_final_load[:, 0], 3).tolist())
print("final eMBB UE counts:", best_final_ue[:, 0].tolist())
print("handovers:", best_handovers)

fig, axes = plt.subplots(1, 4, figsize=(15, 4), constrained_layout=True)
heatmap(axes[0], best_initial_load, "best search: load before", cmap="viridis", vmin=0, vmax=1)
heatmap(axes[1], best_schedule[0], "first bias matrix", cmap="coolwarm", vmin=-1, vmax=1)
heatmap(axes[2], best_final_load, "best search: load after", cmap="viridis", vmin=0, vmax=1)
target_display = np.zeros((3, 3), dtype=float)
target_display[:, 0] = DESIRED_EMBB_LOAD
heatmap(axes[3], best_final_load - target_display, "after - desired eMBB", cmap="coolwarm", vmin=-1, vmax=1)
plt.show()

## Compare Tests Visually

In [ ]:
for name, res in all_results.items():
    fig, axes = plt.subplots(1, 5, figsize=(18, 4), constrained_layout=True)
    heatmap(axes[0], res["load_before"], f"{name}\nload before", cmap="viridis", vmin=0, vmax=1)
    heatmap(axes[1], res["bias_matrix"], "bias", cmap="coolwarm", vmin=-1, vmax=1)
    heatmap(axes[2], offset_matrix_for_target(res["offsets"], 1), "offsets to gNB1", cmap="coolwarm", vmin=-6, vmax=6, fmt=".0f")
    heatmap(axes[3], offset_matrix_for_target(res["offsets"], 2), "offsets to gNB2", cmap="coolwarm", vmin=-6, vmax=6, fmt=".0f")
    heatmap(axes[4], res["load_after"], "load after", cmap="viridis", vmin=0, vmax=1)
    plt.show()